# Stage 2 on Colab: test-time adaptation on CIFAR-10-C

Run this on a free **T4 GPU** (Runtime → Change runtime type → T4 GPU).

* CIFAR-10-C (2.9 GB) and RobustBench weights are cached in Google Drive, so they download only once.
* Every run saves its predictions to Drive (`MyDrive/cmpe252/results`). If the session stops, nothing is lost:
  re-run the setup cells and continue with the next experiment.
* Stage 1 checkpoints go in `MyDrive/cmpe252/checkpoints/` (for example `standard_s1.pt`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cmpe252'
!mkdir -p $DRIVE/data $DRIVE/models $DRIVE/results $DRIVE/checkpoints
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = 'member2/stage2-adaptation-eval'   # change to 'main' after the merge
%cd /content
!rm -rf Augmentation-Test-time-Adaptation
!git clone -q -b $BRANCH https://github.com/KirtanPatel30/Augmentation-Test-time-Adaptation.git
%cd /content/Augmentation-Test-time-Adaptation
!pip -q install git+https://github.com/RobustBench/robustbench.git
# Point data/, models/, results/ at Drive so they survive a disconnect.
!ln -sfn $DRIVE/data data && ln -sfn $DRIVE/models models && ln -sfn $DRIVE/results results && ln -sfn $DRIVE/checkpoints checkpoints

In [ ]:
# Download CIFAR-10-C to Drive once (about 5 minutes the first time).
!bash scripts/download_cifar10c.sh data
!python -m pytest -q tests

## E1: pipeline checks against published numbers

Target severity-5 error (Tent repo): Standard WRN-28-10 **43.5 / 20.4 / 18.6**, AugMix WRN-40-2 **18.3 / 14.5 / 12.1** (none / BN / Tent).

In [ ]:
for model in ['rb_standard_wrn28', 'rb_augmix_wrn40']:
    for method in ['none', 'bn', 'tent']:
        !python eval.py --model $model --method $method --protocol episodic --severities 5

In [ ]:
!python analyze.py --no-figures

## Tent learning rate (4 extra corruptions only)

In [ ]:
for lr in ['1e-4', '3e-4', '1e-3', '3e-3']:
    !python eval.py --model rb_standard_wrn28 --method tent --corruptions extra --severities 5 --lr $lr --tag tune --out-dir results/tune
!python analyze.py --results results/tune --no-figures

## E3: episodic, all 75 conditions, on Stage 1 models

In [ ]:
import glob
CKPTS = sorted(glob.glob('checkpoints/*.pt'))
print(CKPTS)
for ckpt in CKPTS:
    for method in ['none', 'bn', 'tent']:
        !python eval.py --model $ckpt --method $method --protocol episodic --clean

## E4: difficult streams (main experiment)

In [ ]:
for ckpt in CKPTS:
    for method in ['bn', 'tent']:
        !python eval.py --model $ckpt --method $method --protocol continual
        !python eval.py --model $ckpt --method $method --protocol small_batch
    # 3 shuffle orders for the severity-5 Tent runs
    for seed in [1, 2, 3]:
        !python eval.py --model $ckpt --method tent --protocol episodic --severities 5 --shuffle-seed $seed

In [ ]:
!python analyze.py
import pandas as pd
pd.read_csv('results/tables/runs.csv')